# New Adjustment Engine — Live Demo

**Audience:** business / back-office users.  
**Goal:** show that one click in the Streamlit app fans out into a fully traceable, auditable adjustment across the data warehouse.

---

## Opening

> What you're about to see is a single adjustment making its way through three layers of the platform.
>
> In plain terms: a risk manager opens the app, picks a book, enters a scale factor, hits **Submit**. From their point of view, that's one click. Behind that click, the platform does three things.
>
> 1. It **records the request** — who asked for what, on which COB, against which scope. That's our orchestration table, `ADJ_HEADER`. Think of it as the ticket.
> 2. It **registers the adjustment officially** — once the pipeline picks it up, a permanent row is created in `DIMENSION.ADJUSTMENT`. This is the dimension that every downstream consumer — Power BI, reports, audit — joins against. Every adjustment the business ever makes has exactly one row here.
> 3. It **applies the numbers** — the pipeline then writes the actual adjusted measure rows into the fact table for the relevant scope: `FACT.VAR_MEASURES_ADJUSTMENT`, `FACT.STRESS_MEASURES_ADJUSTMENT`, `FACT.SENSITIVITY_MEASURES_ADJUSTMENT`, and so on. One request can fan out to thousands of fact rows — and that's the point. The user doesn't touch a single row by hand.
>
> The takeaway: **one submission, one dimension record, many fact rows — fully traceable end-to-end.** Let me show you.

## Setup

Set the context and the demo filters. Change `DEMO_COBID` and `DEMO_USERNAME` to match the adjustment you just submitted in the Streamlit app.

In [ ]:
USE DATABASE DVLP_RAPTOR_NEWADJ;
USE SCHEMA   ADJUSTMENT_APP;

SET DEMO_COBID    = 20260420;
SET DEMO_USERNAME = 'marcos.magri@gmail.com';

## 1. The ticket — what the user submitted

> This is what the Streamlit form wrote. No numbers have been moved yet — just the request. Notice `RUN_STATUS` — as the pipeline picks it up, this moves from *Pending* → *Running* → *Processed*.

In [ ]:
SELECT  ADJ_ID,
        PROCESS_TYPE,
        ADJUSTMENT_TYPE,
        ENTITY_CODE,
        BOOK_CODE,
        SCALE_FACTOR,
        RUN_STATUS,
        DIMENSION_ADJ_ID,
        CREATED_DATE,
        PROCESS_DATE
FROM    ADJUSTMENT_APP.ADJ_HEADER
WHERE   COBID      = $DEMO_COBID
  AND   USERNAME   = $DEMO_USERNAME
  AND   IS_DELETED = FALSE
ORDER BY CREATED_DATE DESC;

## 2. The official record — one row per processed adjustment

> Once the pipeline picked the ticket up, this row appeared in the dimension. The `ADJUSTMENT_ID` you see here is the permanent key every downstream report joins against.

In [ ]:
SELECT  da.ADJUSTMENT_ID,
        da.COBID,
        da.PROCESS_TYPE,
        da.ADJUSTMENT_TYPE,
        da.ENTITY_CODE,
        da.BOOK_CODE,
        da.SCALE_FACTOR,
        da.RUN_STATUS,
        da.RECORD_COUNT,
        da.CREATED_DATE,
        da.PROCESS_DATE,
        da.REASON
FROM    DIMENSION.ADJUSTMENT da
WHERE   da.COBID    = $DEMO_COBID
  AND   da.USERNAME = $DEMO_USERNAME
ORDER BY da.ADJUSTMENT_ID DESC;

## 3. The actual adjusted rows — VaR scope

> Here's where it stops being metadata and starts being money. Each row is a position whose VaR has been shifted by the adjustment.

In [ ]:
SELECT  fa.ADJUSTMENT_ID,
        fa.COBID,
        fa.ENTITY_CODE,
        fa.CURRENCY_CODE,
        fa.PNL_VECTOR_VALUE,
        fa.PNL_VECTOR_VALUE_IN_USD
FROM    FACT.VAR_MEASURES_ADJUSTMENT fa
JOIN    DIMENSION.ADJUSTMENT         da
  ON    da.ADJUSTMENT_ID = fa.ADJUSTMENT_ID
WHERE   da.COBID    = $DEMO_COBID
  AND   da.USERNAME = $DEMO_USERNAME
ORDER BY fa.ADJUSTMENT_ID DESC
LIMIT 20;

## 4. One ticket → many fact rows — the fan-out

> This is the number that matters. One click, N adjusted rows, across every scope the pipeline touched.

In [ ]:
SELECT  da.ADJUSTMENT_ID,
        da.PROCESS_TYPE,
        da.ENTITY_CODE,
        da.BOOK_CODE,
        da.SCALE_FACTOR,
        COUNT(fa_var.ADJUSTMENT_ID)  AS VAR_ROWS,
        COUNT(fa_str.ADJUSTMENT_ID)  AS STRESS_ROWS,
        COUNT(fa_sen.ADJUSTMENT_ID)  AS SENSITIVITY_ROWS
FROM    DIMENSION.ADJUSTMENT                   da
LEFT JOIN FACT.VAR_MEASURES_ADJUSTMENT          fa_var ON fa_var.ADJUSTMENT_ID = da.ADJUSTMENT_ID
LEFT JOIN FACT.STRESS_MEASURES_ADJUSTMENT       fa_str ON fa_str.ADJUSTMENT_ID = da.ADJUSTMENT_ID
LEFT JOIN FACT.SENSITIVITY_MEASURES_ADJUSTMENT  fa_sen ON fa_sen.ADJUSTMENT_ID = da.ADJUSTMENT_ID
WHERE   da.COBID    = $DEMO_COBID
  AND   da.USERNAME = $DEMO_USERNAME
GROUP BY 1, 2, 3, 4, 5
ORDER BY da.ADJUSTMENT_ID DESC;

## 5. Before and after — the scale factor made visible

> The scale factor isn't an abstraction. Here's the original VaR alongside the adjustment delta for the same positions, so you can see what the number really moved.

In [ ]:
SELECT  da.ADJUSTMENT_ID,
        da.ENTITY_CODE,
        da.BOOK_CODE,
        da.SCALE_FACTOR,
        SUM(m.PNL_VECTOR_VALUE_IN_USD)       AS ORIGINAL_USD,
        SUM(fa.PNL_VECTOR_VALUE_IN_USD)      AS ADJUSTMENT_DELTA_USD,
        SUM(m.PNL_VECTOR_VALUE_IN_USD)
          + SUM(fa.PNL_VECTOR_VALUE_IN_USD)  AS ADJUSTED_USD
FROM    DIMENSION.ADJUSTMENT          da
JOIN    FACT.VAR_MEASURES_ADJUSTMENT  fa ON fa.ADJUSTMENT_ID = da.ADJUSTMENT_ID
JOIN    FACT.VAR_MEASURES             m
  ON    m.COBID         = fa.COBID
  AND   m.ENTITY_CODE   = fa.ENTITY_CODE
  AND   m.BOOK_KEY      = fa.BOOK_KEY
  AND   m.TRADE_KEY     = fa.TRADE_KEY
  AND   m.CURRENCY_CODE = fa.CURRENCY_CODE
WHERE   da.COBID        = $DEMO_COBID
  AND   da.USERNAME     = $DEMO_USERNAME
  AND   da.PROCESS_TYPE = 'VaR'
GROUP BY 1, 2, 3, 4
ORDER BY da.ADJUSTMENT_ID DESC;

## Closing

> So — one submission through Streamlit, one permanent row in `DIMENSION.ADJUSTMENT`, and the fact tables for every scope touched, all keyed by the same `ADJUSTMENT_ID`. From the business user's click, to the audit trail, to the downstream Power BI dashboards — it's the same story, told once, in the warehouse.
>
> Happy to take questions.